# Audytor WCAG – moduł AI: ocena tekstu alternatywnego (1.1.1)
**Sprint 1, osoba 3.** Bielik 11B v3.0 (Ollama) na Colabie + moduł `ocen_alt` + ewaluacja na przykładach.

**Zanim zaczniesz:** `Środowisko wykonawcze → Zmień typ środowiska → T4 GPU`.

Moduł łączy się z modelem przez API zgodne z OpenAI. Zmiana `BASE_URL` / `MODEL` w sekcji 3 przełącza backend (Ollama lokalnie, PLGrid, CloudFerro, model komercyjny) bez zmian w kodzie – przyda się w F16.

## 1. Ollama + Bielik

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# zstd jest wymagany przez instalator Ollamy, pciutils/lshw do wykrycia GPU
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests, os

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
ollama_proc = subprocess.Popen(["ollama", "serve"],
                               stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=1); print("Ollama działa"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama nie wstała – sprawdź ollama.log")

In [ ]:
# Oficjalny model SpeakLeash. Q4_K_M ~6.7 GB. Na T4 (15 GB) zmieści się też :Q8_0 (~12 GB) – lepsza jakość, wolniej.
MODEL_TAG = "SpeakLeash/bielik-11b-v3.0-instruct:Q4_K_M"
!ollama pull {MODEL_TAG}

## 2. Zależności

In [ ]:
!pip -q install openai pydantic pandas

## 3. Konfiguracja backendu (jedyne miejsce do zmiany)

In [ ]:
BASE_URL = "http://127.0.0.1:11434/v1"   # Ollama; np. PLGrid: https://llmlab.plgrid.pl/api/v1
API_KEY  = "ollama"                        # Ollama ignoruje klucz; dla innych backendów wstaw prawdziwy
MODEL    = MODEL_TAG
TEMPERATURE = 0.1                          # nisko = powtarzalnie

from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

t = time.time()
r = client.chat.completions.create(model=MODEL, temperature=TEMPERATURE,
        messages=[{"role": "user", "content": "Odpowiedz jednym zdaniem: czym jest WCAG?"}])
print(r.choices[0].message.content)
print(f"{time.time()-t:.1f} s (pierwsze wywołanie ładuje model do VRAM, jest wolniejsze)")

## 4. Moduł `ocen_alt`
Najpierw tanie reguły (brak atrybutu, nazwa pliku, słowa-wypełniacze). Model dostaje tylko przypadki, których reguły nie rozstrzygają.

Model **klasyfikuje typ problemu**, a werdykt wylicza kod. Dzięki temu zachowanie jest powtarzalne, a mapowanie łatwo zmienić bez ruszania promptu.

**Ograniczenie:** bez informacji o obrazie model ocenia alt tylko na podstawie kontekstu. Parametr `obraz_opis` jest na wynik modelu wizyjnego (osoba 4).

In [ ]:
import re, json
from dataclasses import dataclass, asdict
from typing import Optional, Literal

Kategoria = Literal["ok", "pusty_dekoracyjny", "pusty_informacyjny", "brak_atrybutu",
                    "nazwa_pliku", "zbyt_ogolny", "powtorzenie_kontekstu",
                    "niezgodny_z_obrazem", "wyglad_zamiast_funkcji", "niepewne"]

WERDYKT = {  # kategoria -> werdykt (decyzja w kodzie, nie w modelu)
    "ok": "ok",
    "pusty_dekoracyjny": "ok",
    "pusty_informacyjny": "blad",
    "brak_atrybutu": "blad",
    "nazwa_pliku": "blad",
    "zbyt_ogolny": "blad",
    "powtorzenie_kontekstu": "do_weryfikacji",
    "niezgodny_z_obrazem": "blad",
    "wyglad_zamiast_funkcji": "blad",
    "niepewne": "do_weryfikacji",
}

@dataclass
class WynikAlt:
    werdykt: str            # ok | blad | do_weryfikacji
    kategoria: str
    pewnosc: float          # 0..1
    uzasadnienie: str
    zrodlo: str             # "reguła" | "ai"
    kryterium: str = "1.1.1"

RE_PLIK = re.compile(r"^\s*[\w\-. ]+\.(jpe?g|png|gif|webp|svg|bmp|avif)\s*$", re.I)
RE_TECH = re.compile(r"^\s*(img|image|dsc|pic|photo|screenshot)[\s_\-]*\d+\s*$", re.I)
WYPELNIACZE = {"zdjęcie", "zdjecie", "obrazek", "obraz", "grafika", "foto", "fotografia",
               "ikona", "image", "img", "picture", "photo", "logo", "baner", "banner"}
# Słowa opisujące WYGLĄD ikony. Reguła tylko sygnalizuje (do_weryfikacji), bo obok może być
# widoczny tekst – ostateczną ocenę zostawiamy człowiekowi.
IKONY = {"lupa", "strzałka", "strzalka", "strzałka w prawo", "strzałka w lewo", "koperta",
         "domek", "dom", "drukarka", "telefon", "słuchawka", "kosz", "plus", "minus", "krzyżyk",
         "x", "hamburger", "trzy kreski", "zębatka", "kłódka", "ludzik", "serce", "gwiazdka"}

def reguly(alt: Optional[str], w_linku: bool = False) -> Optional[WynikAlt]:
    if alt is None:
        return WynikAlt("blad", "brak_atrybutu", 1.0, "Element <img> nie ma atrybutu alt.", "reguła")
    a = alt.strip()
    if a and (RE_PLIK.match(a) or RE_TECH.match(a)):
        return WynikAlt("blad", "nazwa_pliku", 0.95, f"Alt wygląda jak nazwa pliku: „{a}”.", "reguła")
    if a.lower().strip(".!") in WYPELNIACZE:
        return WynikAlt("blad", "zbyt_ogolny", 0.9, f"Alt „{a}” nie opisuje treści obrazu.", "reguła")
    if w_linku and a.lower().strip(".!") in IKONY:
        return WynikAlt("do_weryfikacji", "wyglad_zamiast_funkcji", 0.7,
                        f"Alt „{a}” opisuje wygląd ikony, a nie funkcję linku/przycisku.", "reguła")
    return None  # rozstrzyga model

SYSTEM = """Jesteś audytorem dostępności cyfrowej (WCAG 2.1, kryterium 1.1.1 „Treść nietekstowa”).
Oceniasz tekst alternatywny (atrybut alt) obrazu na polskiej stronie internetowej.

Wybierz DOKŁADNIE JEDNĄ kategorię:
- ok – alt zwięźle przekazuje treść obrazu albo, gdy obraz jest w linku/przycisku, jego cel lub funkcję
- pusty_dekoracyjny – alt jest pusty, a obraz jest czysto dekoracyjny (to POPRAWNE)
- pusty_informacyjny – alt jest pusty, a obraz niesie informację lub jest jedyną treścią linku/przycisku
- zbyt_ogolny – alt nic nie mówi o treści (np. „zdjęcie”, „grafika artykułu”)
- powtorzenie_kontekstu – alt powtarza tekst stojący tuż obok obrazu
- niezgodny_z_obrazem – alt nie pasuje do opisu obrazu (tylko gdy podano opis obrazu)
- wyglad_zamiast_funkcji – obraz jest w linku/przycisku, a alt NAZYWA PRZEDMIOT LUB KSZTAŁT
  narysowany na ikonie (np. „lupa”, „strzałka”, „koperta”, „symbol drukarki”), zamiast celu lub funkcji
- niepewne – brak wystarczających informacji do oceny

Pytanie kontrolne przed wyborem wyglad_zamiast_funkcji:
Czy alt nazywa rzecz, którą WIDAĆ na obrazku? Jeśli NIE – to nie jest wyglad_zamiast_funkcji.
Alt nazywający miejsce docelowe lub akcję jest poprawny, także bez czasownika, np.:
„Strona główna”, „Następny slajd”, „Urząd Gminy X – strona główna”, „Szukaj”, „Napisz do nas”.
Przykład błędu: ikona w linku do formularza kontaktowego z alt „Ikona koperty” -> wyglad_zamiast_funkcji.

Nie zgaduj, co jest na obrazie, jeśli nie podano jego opisu.
Odpowiedz WYŁĄCZNIE obiektem JSON:
{"kategoria": "...", "pewnosc": liczba 0-1, "uzasadnienie": "jedno zdanie po polsku"}"""

def _prompt(alt, kontekst, src, w_linku, obraz_opis):
    czesci = [f'Alt: "{alt}"' if alt else "Alt: (pusty)",
              f"Plik: {src}" if src else None,
              f"Obraz jest w linku/przycisku: {'tak' if w_linku else 'nie'}",
              f"Tekst wokół obrazu: {kontekst or '(brak)'}",
              f"Opis obrazu (model wizyjny): {obraz_opis}" if obraz_opis else None]
    return "\n".join(c for c in czesci if c)

def _parsuj(tekst: str) -> dict:
    m = re.search(r"\{.*\}", tekst, re.S)  # na wypadek komentarza wokół JSON-a
    d = json.loads(m.group(0) if m else tekst)
    if d.get("kategoria") not in WERDYKT:
        d["kategoria"] = "niepewne"
    d["pewnosc"] = min(max(float(d.get("pewnosc", 0.5)), 0.0), 1.0)
    return d

def ocen_alt(alt: Optional[str], kontekst: str = "", src: str = "",
             w_linku: bool = False, obraz_opis: Optional[str] = None) -> WynikAlt:
    if (w := reguly(alt, w_linku)):
        return w
    try:
        r = client.chat.completions.create(
            model=MODEL, temperature=TEMPERATURE,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": _prompt(alt, kontekst, src, w_linku, obraz_opis)}])
        d = _parsuj(r.choices[0].message.content)
    except Exception as e:
        return WynikAlt("do_weryfikacji", "niepewne", 0.0, f"Błąd modelu: {e}", "ai")
    werdykt = WERDYKT[d["kategoria"]]
    if werdykt != "ok" and d["pewnosc"] < 0.6:   # niska pewność -> do człowieka
        werdykt = "do_weryfikacji"
    return WynikAlt(werdykt, d["kategoria"], d["pewnosc"], d.get("uzasadnienie", ""), "ai")

In [ ]:
asdict(ocen_alt("Kliknij", kontekst="Zobacz nasz nowy cennik usług", w_linku=True))

## 5. Zestaw testowy
18 przykładów (sprint 1) – **rozbuduj o alty z prawdziwych migawek** (zadanie ze sprintu 1). Dbaj o przypadki graniczne: logo w linku, wykres, ikona-przycisk, długi alt, alt po angielsku.

`oczekiwany` = werdykt, który dałby człowiek-audytor.

In [ ]:
PRZYKLADY = [
  {"alt": None, "kontekst": "Aktualności gminy", "src": "news.jpg", "w_linku": False, "oczekiwany": "blad"},
  {"alt": "IMG_0423.jpg", "kontekst": "Dożynki gminne 2026", "src": "IMG_0423.jpg", "w_linku": False, "oczekiwany": "blad"},
  {"alt": "zdjęcie", "kontekst": "Nowy budynek przedszkola", "src": "p1.jpg", "w_linku": False, "oczekiwany": "blad"},
  {"alt": "", "kontekst": "Ozdobny szlaczek oddzielający sekcje strony", "src": "separator.png", "w_linku": False, "oczekiwany": "ok"},
  {"alt": "", "kontekst": "(obraz jest jedyną zawartością linku do strony głównej)", "src": "logo-gmina.png", "w_linku": True, "oczekiwany": "blad"},
  {"alt": "Urząd Gminy Supraśl – strona główna", "kontekst": "Nagłówek strony", "src": "logo.png", "w_linku": True, "oczekiwany": "ok"},
  {"alt": "Wykres", "kontekst": "Liczba mieszkańców gminy w latach 2015–2025 spadła z 15 200 do 14 100", "src": "wykres.png", "w_linku": False, "oczekiwany": "blad"},
  {"alt": "Wójt gminy wręcza nagrodę laureatce konkursu plastycznego", "kontekst": "Rozstrzygnięcie konkursu plastycznego", "src": "konkurs.jpg", "w_linku": False, "oczekiwany": "ok"},
  {"alt": "Rekrutacja do przedszkoli 2026/2027", "kontekst": "Rekrutacja do przedszkoli 2026/2027", "src": "rekrutacja.jpg", "w_linku": False, "oczekiwany": "do_weryfikacji"},
  {"alt": "Lupa", "kontekst": "(ikona w przycisku wysyłającym formularz wyszukiwania)", "src": "search.svg", "w_linku": True, "oczekiwany": "blad"},
  # nowe: wygląd zamiast funkcji, w sformułowaniach, których NIE ma na liście reguł
  {"alt": "Szkło powiększające", "kontekst": "(ikona w przycisku formularza wyszukiwania)", "src": "szukaj.svg", "w_linku": True, "oczekiwany": "blad"},
  {"alt": "Ikona strzałki w prawo", "kontekst": "(przycisk przewijający karuzelę aktualności do następnego slajdu)", "src": "arrow-right.svg", "w_linku": True, "oczekiwany": "blad"},
  {"alt": "Symbol drukarki", "kontekst": "(link otwierający wersję strony do druku)", "src": "print.png", "w_linku": True, "oczekiwany": "blad"},
  # kontrprzykład: alt opisuje funkcję -> ok
  {"alt": "Następny slajd", "kontekst": "(przycisk przewijający karuzelę aktualności)", "src": "next.svg", "w_linku": True, "oczekiwany": "ok"},
  # kontrprzykłady spoza promptu (sprawdzają uogólnienie)
  {"alt": "Poprzednie zdjęcie", "kontekst": "(przycisk w galerii zdjęć z dożynek)", "src": "prev.svg", "w_linku": True, "oczekiwany": "ok"},
  {"alt": "Facebook gminy", "kontekst": "(ikona w stopce prowadząca do profilu gminy na Facebooku)", "src": "fb.svg", "w_linku": True, "oczekiwany": "ok"},
  {"alt": "Niebieskie logo Facebooka", "kontekst": "(ikona w stopce prowadząca do profilu gminy na Facebooku)", "src": "fb.svg", "w_linku": True, "oczekiwany": "blad"},
  {"alt": "Zamknij", "kontekst": "(przycisk w rogu okna z komunikatem o cookies)", "src": "close.svg", "w_linku": True, "oczekiwany": "ok"},
]

with open("alt_testy.jsonl", "w", encoding="utf-8") as f:
    for p in PRZYKLADY:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
len(PRZYKLADY)

## 6. Ewaluacja

In [ ]:
import pandas as pd

wiersze = []
for p in PRZYKLADY:
    t = time.time()
    w = ocen_alt(p["alt"], p["kontekst"], p["src"], p["w_linku"])
    wiersze.append({**p, **asdict(w), "czas_s": round(time.time() - t, 2)})

df = pd.DataFrame(wiersze)
df["trafiony"] = df["werdykt"] == df["oczekiwany"]
# Wykrycie: czy problem w ogóle trafił do człowieka (blad LUB do_weryfikacji), niezależnie od werdyktu
df["problem_oczekiwany"] = df["oczekiwany"] != "ok"
df["problem_zgloszony"]  = df["werdykt"] != "ok"
pd.set_option("display.max_colwidth", 80)
df[["alt", "oczekiwany", "werdykt", "kategoria", "pewnosc", "zrodlo", "czas_s", "uzasadnienie"]]

In [ ]:
ai = df[df.zrodlo == "ai"]
print(f"Trafność ogółem:      {df.trafiony.mean():.0%}  ({df.trafiony.sum()}/{len(df)})")
print(f"Trafność samego AI:   {ai.trafiony.mean():.0%}  ({ai.trafiony.sum()}/{len(ai)})" if len(ai) else "brak przypadków AI")
print(f"Średni czas AI:       {ai.czas_s.mean():.2f} s" if len(ai) else "")
pominiete = df[df.problem_oczekiwany & ~df.problem_zgloszony]   # najgorszy błąd: problem znika z raportu
falszywe  = df[~df.problem_oczekiwany & df.problem_zgloszony]   # fałszywy alarm: dodatkowa praca audytora
print(f"Wykryte problemy:     {(df.problem_oczekiwany & df.problem_zgloszony).sum()}/{df.problem_oczekiwany.sum()}")
print(f"Pominięte problemy:   {len(pominiete)}   Fałszywe alarmy: {len(falszywe)}")
print("Wg źródła:"); print(df.groupby("zrodlo").trafiony.agg(["count", "mean"]).rename(columns={"mean": "trafnosc"}))
print()
print(pd.crosstab(df.oczekiwany, df.werdykt, rownames=["oczekiwany"], colnames=["werdykt"]))

In [ ]:
print("POMYŁKI:")
for _, r in df[~df.trafiony].iterrows():
    print(f"- {r.alt!r}: oczekiwany={r.oczekiwany}, werdykt={r.werdykt}, kat={r.kategoria}, źródło={r.zrodlo}\n  → {r.uzasadnienie}")

## 7. Notatka do Definition of Done
Pełna wersja: `docs/ai-ocena-alt.md`.
- **Model:** Bielik-11B-v3.0-Instruct, kwantyzacja: …
- **Uruchomienie:** Ollama na Google Colab (T4, 15 GB VRAM), API zgodne z OpenAI
- **Trafność na N przykładach:** … ; najczęstsze pomyłki: …
- **Czas odpowiedzi:** ~… s / obraz → strona z 30 obrazami ≈ … s
- **Koszt:** Colab darmowy (sesja wygasa, nie nadaje się jako stały backend); produkcyjnie: PLGrid / CloudFerro / własny GPU
- **Wymagania sprzętowe:** Q4_K_M ~6.7 GB VRAM (wejdzie też na RTX 3070 8 GB), Q8_0 ~12 GB
- **Wnioski / ryzyka:** pewność zwracana przez model nie jest skalibrowana; bez opisu obrazu nie da się sprawdzić zgodności alt z obrazem

In [ ]:
# Opcjonalnie: zapis wyników na Dysk
# from google.colab import drive; drive.mount('/content/drive')
# df.to_csv('/content/drive/MyDrive/alt_wyniki.csv', index=False)
df.to_excel("alt_wyniki.xlsx", index=False)
df.to_csv("alt_wyniki.csv", index=False, sep=";", encoding="utf-8-sig")  # CSV czytelny dla Excela